# Token-level T5 FactorVAE for GoEmotions — readable training notebook

This notebook keeps the **token-level VAE architecture**, the **frozen T5 backbone**, and the **LoRA-only copy pathway**, but reorganizes the workflow so it is easier to read and easier to debug.

The main goals are:

- keep the architecture close to the original intent,
- fix the training and evaluation logic,
- make every epoch easy to inspect,
- show both standard metrics and concrete text-level behavior.


## Run policy

- No AMP is used.
- The backbone stays frozen and only LoRA weights become trainable starting at **epoch 6**.
- The latent is a real bottleneck.
- The VAE-side loss weights stay constant across training.
- Evaluation happens every epoch.
- A small qualitative monitor is printed every epoch on two fixed validation examples.
- Final test evaluation still happens only after training.


## Notebook map

The notebook is intentionally split into small sections:

1. configuration and reproducibility,
2. dataset setup,
3. model blocks,
4. losses and metrics,
5. qualitative monitoring helpers,
6. training loop,
7. final evaluation and optional experiments.

The per-epoch monitor prints, for each of two fixed validation examples:

- the input text,
- the gold and predicted emotions,
- the copy output when bypassing the VAE,
- the copy output when going through the VAE bottleneck,
- the predicted emotions of those generated texts,
- an emotion-editing example with two fixed target emotions.


In [3]:

from __future__ import annotations

from collections import Counter
from dataclasses import asdict, dataclass
import gc
import json
import math
import os
import random
import warnings
from difflib import SequenceMatcher
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import DatasetDict, load_dataset
from peft import LoraConfig, TaskType, get_peft_model
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    f1_score,
    hamming_loss,
    jaccard_score,
    label_ranking_average_precision_score,
    precision_recall_fscore_support,
)
from torch.utils.data import DataLoader, Dataset, Subset
from tqdm.auto import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from transformers.modeling_outputs import BaseModelOutput


## Configuration

The defaults below keep the token-level design but make the training schedule easier to interpret.

Two choices are especially important:

- `max_length=128` is used everywhere for consistency.
- the KL term is kept deliberately light and warmed up slowly, because the raw KL value is not on the same scale as the other losses.


In [ ]:
@dataclass
class DataConfig:
    dataset_repo: str = "google-research-datasets/go_emotions"
    dataset_config: str = "simplified"
    max_length: int = 128
    train_batch_size: int = 32
    eval_batch_size: int = 64
    num_workers: int = 0
    calibration_fraction: float = 0.05


@dataclass
class PromptConfig:
    use_prompt: bool = False
    prompt_text: str = "I am exactly repeating: "
    mask_prompt_loss: bool = True
    max_decoder_length: int = 128


@dataclass
class ModelConfig:
    model_name: str = "google/flan-t5-base"
    num_scalar_factors: int = 28
    vector_latent_dim: int = 256
    vae_hidden_dim: int = 1024
    residual_bottleneck_dim: int = 2
    residual_bottleneck_dropout: float = 0.1
    use_lora: bool = True
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.1
    lora_target_modules: Tuple[str, ...] = ("q", "v")
    latent_pool_heads: int = 8
    attention_source: str = "encoder_sequence"
    pooling_mode: str = "per_scalar_dim"
    classifier_mode: str = "per_emotion_mlp"
    classifier_dropout: float = 0.1
    per_emotion_hidden_dim: int = 32
    joint_mlp_hidden_dim: int = 128
    classifier_uses_mu: bool = True
    use_skip_connection: bool = False
    vae_dropout: float = 0.1
    vector_adversary_hidden_dim: int = 128

In [ ]:
@dataclass
class LossConfig:
    classification_weight: float = 4.0
    recon_weight: float = 2.0
    kl_weight: float = 1.0
    tc_weight: float = 1.0
    tc_subspace: str = "scalar"
    copy_weight: float = 0.5
    vector_adv_weight: float = 0.1
    vector_sep_weight: float = 0.005


@dataclass
class ScheduleConfig:
    num_epochs: int = 30
    lr: float = 2e-4
    disc_lr: float = 5e-5
    vector_adv_lr: float = 2e-4
    weight_decay: float = 1e-2
    max_grad_norm: float = 1.0
    warmup_ratio: float = 0.1
    min_lr_scale: float = 0.1
    lora_start_epoch: int = 4
    copy_loss_start_epoch: int = 4
    eval_every_epoch: bool = True


@dataclass
class ExperimentConfig:
    seed: int = 42
    output_dir: str = "factorvae_tokenlevel_clean_artifacts"
    sample_posterior_train: bool = True
    sample_posterior_eval: bool = False
    prompt_eval_num_examples: int = 32
    prompt_candidates: Tuple[str, ...] = (
        "",
        "I am exactly repeating: ",
        "Repeat exactly: ",
        "Copy exactly: ",
    )
    eval_num_copy_examples: int = 32
    show_num_examples: int = 2
    monitor_num_examples: int = 2
    monitor_edit_num_labels: int = 2
    monitor_edit_steps: int = 24
    threshold_grid_points: int = 101


DATA_CONFIG = DataConfig()
PROMPT_CONFIG = PromptConfig()
MODEL_CONFIG = ModelConfig()
LOSS_CONFIG = LossConfig()
SCHEDULE_CONFIG = ScheduleConfig()
EXPERIMENT_CONFIG = ExperimentConfig()


In [4]:

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass


set_seed(EXPERIMENT_CONFIG.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("seed:", EXPERIMENT_CONFIG.seed)


device: cuda
seed: 42


## Data loading

The dataset section stays simple and explicit.

This version:

- calls `load_dataset(...)` once,
- keeps raw texts instead of pretokenizing the whole dataset,
- tokenizes inside the collator,
- uses `drop_last=True` for training so the discriminator always sees valid batches,
- reserves a small train-side calibration split for threshold tuning,
- picks two fixed validation examples for the epoch-by-epoch qualitative monitor.


In [5]:

def parse_label_ids(label_value: Any) -> List[int]:
    if label_value is None:
        return []
    if isinstance(label_value, (list, tuple, set)):
        return [int(x) for x in label_value]
    if isinstance(label_value, str):
        text = label_value.strip()
        if not text:
            return []
        if text.startswith("[") and text.endswith("]"):
            text = text[1:-1]
        return [int(x.strip()) for x in text.split(",") if x.strip()]
    return [int(label_value)]


class GoEmotionsExampleDataset(Dataset):
    def __init__(self, hf_split, num_labels: int) -> None:
        self.texts = [str(x) for x in hf_split["text"]]
        labels: List[torch.Tensor] = []
        for raw_labels in hf_split["labels"]:
            target = torch.zeros(num_labels, dtype=torch.float32)
            for idx in parse_label_ids(raw_labels):
                if 0 <= idx < num_labels:
                    target[idx] = 1.0
            labels.append(target)
        self.labels = torch.stack(labels, dim=0)

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, index: int) -> Dict[str, Any]:
        return {
            "text": self.texts[index],
            "labels": self.labels[index],
        }


class GoEmotionsCollator:
    def __init__(self, tokenizer: AutoTokenizer, max_length: int) -> None:
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __call__(self, batch: Sequence[Dict[str, Any]]) -> Dict[str, Any]:
        texts = [item["text"] for item in batch]
        labels = torch.stack([item["labels"] for item in batch], dim=0)
        tokenized = self.tokenizer(
            texts,
            max_length=self.max_length,
            padding=True,
            truncation=True,
            return_tensors="pt",
        )
        return {
            "texts": texts,
            "input_ids": tokenized["input_ids"],
            "attention_mask": tokenized["attention_mask"],
            "labels": labels,
        }


raw_dataset: DatasetDict = load_dataset(DATA_CONFIG.dataset_repo, DATA_CONFIG.dataset_config)
train_split = raw_dataset["train"]
val_split = raw_dataset["validation"] if "validation" in raw_dataset else raw_dataset["test"]
test_split = raw_dataset["test"] if "test" in raw_dataset else val_split
emotion_names = list(train_split.features["labels"].feature.names)
num_labels = len(emotion_names)

print("dataset repo:", DATA_CONFIG.dataset_repo)
print("dataset config:", DATA_CONFIG.dataset_config)
print("num labels:", num_labels)
print("first five labels:", emotion_names[:5])


dataset repo: google-research-datasets/go_emotions
dataset config: simplified
num labels: 28
first five labels: ['admiration', 'amusement', 'anger', 'annoyance', 'approval']


In [6]:

tokenizer = AutoTokenizer.from_pretrained(MODEL_CONFIG.model_name)

train_dataset = GoEmotionsExampleDataset(train_split, num_labels=num_labels)
val_dataset = GoEmotionsExampleDataset(val_split, num_labels=num_labels)
test_dataset = GoEmotionsExampleDataset(test_split, num_labels=num_labels)

all_train_indices = np.arange(len(train_dataset))
rng = np.random.default_rng(EXPERIMENT_CONFIG.seed)
rng.shuffle(all_train_indices)
num_calibration = max(1, int(len(all_train_indices) * DATA_CONFIG.calibration_fraction))
calibration_indices = all_train_indices[:num_calibration].tolist()
train_core_indices = all_train_indices[num_calibration:].tolist()

train_core_dataset = Subset(train_dataset, train_core_indices)
threshold_tune_dataset = Subset(train_dataset, calibration_indices)

collator = GoEmotionsCollator(tokenizer=tokenizer, max_length=DATA_CONFIG.max_length)

generator = torch.Generator()
generator.manual_seed(EXPERIMENT_CONFIG.seed)

train_loader = DataLoader(
    train_core_dataset,
    batch_size=DATA_CONFIG.train_batch_size,
    shuffle=True,
    drop_last=True,
    num_workers=DATA_CONFIG.num_workers,
    collate_fn=collator,
    generator=generator,
)
threshold_loader = DataLoader(
    threshold_tune_dataset,
    batch_size=DATA_CONFIG.eval_batch_size,
    shuffle=False,
    drop_last=False,
    num_workers=DATA_CONFIG.num_workers,
    collate_fn=collator,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=DATA_CONFIG.eval_batch_size,
    shuffle=False,
    drop_last=False,
    num_workers=DATA_CONFIG.num_workers,
    collate_fn=collator,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=DATA_CONFIG.eval_batch_size,
    shuffle=False,
    drop_last=False,
    num_workers=DATA_CONFIG.num_workers,
    collate_fn=collator,
)


def compute_pos_weight(label_matrix: torch.Tensor, max_ratio: float = 20.0) -> torch.Tensor:
    pos = label_matrix.sum(dim=0)
    neg = label_matrix.size(0) - pos
    return (neg / pos.clamp_min(1.0)).clamp(max=max_ratio)


pos_weight = compute_pos_weight(train_dataset.labels)
print("train core samples:", len(train_core_dataset))
print("threshold-tune samples:", len(threshold_tune_dataset))
print("train batches:", len(train_loader))
print("threshold batches:", len(threshold_loader))
print("val batches:", len(val_loader))
print("test batches:", len(test_loader))
print(
    "pos_weight stats:",
    {
        "min": round(float(pos_weight.min().item()), 3),
        "mean": round(float(pos_weight.mean().item()), 3),
        "max": round(float(pos_weight.max().item()), 3),
    },
)


train core samples: 41240
threshold-tune samples: 2170
train batches: 5155
threshold batches: 272
val batches: 679
test batches: 679
pos_weight stats: {'min': 2.053, 'mean': 18.339, 'max': 20.0}


## Fixed qualitative monitor examples

The examples below are sampled once from the validation split using the global seed.

They stay fixed across epochs so you can compare progress directly instead of reading different examples every time.


In [7]:
monitor_rng = random.Random(EXPERIMENT_CONFIG.seed + 7)
monitor_example_indices = monitor_rng.sample(
    list(range(len(val_dataset))),
    k=min(EXPERIMENT_CONFIG.monitor_num_examples, len(val_dataset)),
)
monitor_target_labels = [
    monitor_rng.sample(list(emotion_names), k=EXPERIMENT_CONFIG.monitor_edit_num_labels)
    for _ in monitor_example_indices
]

print("monitor example indices:", monitor_example_indices)
for idx, target_labels in zip(monitor_example_indices, monitor_target_labels):
    preview = val_dataset[idx]["text"]
    print(f"- val index {idx}: edit targets={target_labels} | text={preview[:90]!r}")


monitor example indices: [547, 2820]
- val index 547: edit targets=['excitement', 'annoyance'] | text='Meh good introduction. Sadly I am a pro philosopher so know all of this'
- val index 2820: edit targets=['disapproval', 'remorse'] | text="if it shatters your little ego i'm fine with this. :)"


## Model outputs and utility helpers

A few utilities are centralized here so the rest of the notebook can stay explicit and less fragile.


In [8]:
@dataclass
class FactorVAEOutput:
    z: torch.Tensor
    mu: torch.Tensor
    logvar: torch.Tensor
    scalar_z: torch.Tensor
    vector_z: torch.Tensor
    scalar_mu: torch.Tensor
    vector_mu: torch.Tensor
    scalar_logvar: torch.Tensor
    vector_logvar: torch.Tensor


@dataclass
class T5FactorVAEModelOutput:
    t5_encoder_sequence: torch.Tensor
    attention_weights: torch.Tensor
    pooled_scalar_tensor: torch.Tensor
    vae: FactorVAEOutput
    vae_decoded_sequence: torch.Tensor
    decoder_memory: torch.Tensor
    classification_logits: torch.Tensor
    base_loss: Optional[torch.Tensor] = None
    loss_terms: Optional[Dict[str, torch.Tensor]] = None


@dataclass
class TrainingState:
    model: nn.Module
    discriminator: Optional[nn.Module]
    vector_adversary: Optional[nn.Module]
    optimizer: torch.optim.Optimizer
    disc_optimizer: Optional[torch.optim.Optimizer]
    vector_adv_optimizer: Optional[torch.optim.Optimizer]
    lr_scheduler: torch.optim.lr_scheduler.LambdaLR
    threshold: float
    best_val_micro_f1: float
    step: int
    history: List[Dict[str, Any]]


@dataclass
class ExperimentContext:
    tokenizer: AutoTokenizer
    emotion_names: Sequence[str]
    device: torch.device
    data_config: DataConfig
    prompt_config: PromptConfig
    model_config: ModelConfig
    loss_config: LossConfig
    schedule_config: ScheduleConfig
    experiment_config: ExperimentConfig
    pos_weight: torch.Tensor


In [9]:
def resolve_hidden_size_from_config(config: Any) -> int:
    candidates = [
        config,
        getattr(config, "text_config", None),
        getattr(config, "encoder", None),
        getattr(config, "decoder", None),
    ]
    for cfg in candidates:
        if cfg is None:
            continue
        for key in ("hidden_size", "d_model", "model_dim"):
            value = getattr(cfg, key, None)
            if isinstance(value, int):
                return value
    raise ValueError("Could not resolve hidden size from model config.")



def is_decoder_cross_attention_lora_parameter(name: str) -> bool:
    if "lora_" not in name:
        return False
    lower = name.lower()
    if "decoder.block" not in lower:
        return False
    return ("encdecattention" in lower) or ("layer.1" in lower)



def freeze_non_lora_parameters(module: nn.Module) -> None:
    for name, param in module.named_parameters():
        param.requires_grad = is_decoder_cross_attention_lora_parameter(name)



def set_lora_trainable(module: nn.Module, enabled: bool) -> None:
    for name, param in module.named_parameters():
        if "lora_" in name:
            param.requires_grad = enabled and is_decoder_cross_attention_lora_parameter(name)



def set_requires_grad(module: nn.Module, enabled: bool) -> None:
    for param in module.parameters():
        param.requires_grad = enabled



def shift_tokens_right_manual(
    input_ids: torch.LongTensor,
    pad_token_id: int,
    decoder_start_token_id: int,
) -> torch.LongTensor:
    shifted = input_ids.new_full(input_ids.shape, pad_token_id)
    shifted[:, 0] = decoder_start_token_id
    shifted[:, 1:] = input_ids[:, :-1]
    return shifted



def masked_mean(sequence: torch.Tensor, attention_mask: Optional[torch.Tensor]) -> torch.Tensor:
    if attention_mask is None:
        return sequence.mean(dim=1)
    mask = attention_mask.to(dtype=sequence.dtype).unsqueeze(-1)
    denom = mask.sum(dim=1).clamp_min(1.0)
    return (sequence * mask).sum(dim=1) / denom



def safe_div(numerator: float, denominator: float) -> float:
    return float(numerator / denominator) if abs(denominator) > 1e-12 else 0.0


def release_memory() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()



def to_serializable(value: Any) -> Any:
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().tolist()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.floating, np.integer)):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(k): to_serializable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_serializable(v) for v in value]
    return value



def save_json(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(to_serializable(payload), indent=2))



def optimizable_model_parameters(model: nn.Module) -> List[nn.Parameter]:
    params: List[nn.Parameter] = []
    for name, param in model.named_parameters():
        if name.startswith("shared_t5") and "lora_" not in name:
            continue
        params.append(param)
    return params


## Core architecture

This keeps the original structure:

1. frozen T5 encoder,
2. token-level VAE over encoder hidden states,
3. scalar-latent attention pooling for multilabel prediction,
4. VAE decoder that reconstructs token-level memory,
5. frozen T5 decoder + optional LoRA for copy training.

The important behavioral changes are:

- the latent is smaller than the encoder state,
- the direct encoder skip path is removed and replaced with a learnable residual bottleneck before decoding,
- the default classifier and pooling settings are the interpretable ones,
- the shared T5 model stays in `eval()` mode even during training so frozen dropout does not inject noise.


In [10]:

class ScalarLatentAttentionPooling(nn.Module):
    def __init__(
        self,
        num_scalar_factors: int,
        source_dim: int,
        num_heads: int = 4,
        pooling_mode: str = "per_scalar_dim",
        dropout: float = 0.0,
    ) -> None:
        super().__init__()
        valid_modes = {"per_scalar_dim", "joint_scalar_vector"}
        if pooling_mode not in valid_modes:
            raise ValueError(f"Unsupported pooling_mode={pooling_mode}. Expected one of {sorted(valid_modes)}")

        self.num_scalar_factors = num_scalar_factors
        self.source_dim = source_dim
        self.num_heads = num_heads
        self.pooling_mode = pooling_mode
        self.dropout = nn.Dropout(dropout)

        if pooling_mode == "per_scalar_dim":
            self.score_weight = nn.Parameter(torch.empty(num_scalar_factors, num_heads, source_dim))
            self.score_bias = nn.Parameter(torch.zeros(num_scalar_factors, num_heads))
        else:
            self.score_weight = nn.Parameter(torch.empty(num_heads, source_dim))
            self.score_bias = nn.Parameter(torch.zeros(num_heads))
        nn.init.xavier_uniform_(self.score_weight)

    def forward(
        self,
        source_sequence: torch.Tensor,
        scalar_sequence: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        source_sequence = self.dropout(source_sequence)

        if self.pooling_mode == "per_scalar_dim":
            logits = torch.einsum("bsd,nhd->bnhs", source_sequence, self.score_weight)
            logits = logits + self.score_bias.unsqueeze(0).unsqueeze(-1)
        else:
            shared_logits = torch.einsum("bsd,hd->bhs", source_sequence, self.score_weight)
            shared_logits = shared_logits + self.score_bias.unsqueeze(0).unsqueeze(-1)
            logits = shared_logits.unsqueeze(1).expand(-1, self.num_scalar_factors, -1, -1)

        if attention_mask is not None:
            keep_mask = attention_mask.bool()
            logits = logits.masked_fill(
                ~keep_mask.unsqueeze(1).unsqueeze(1),
                torch.finfo(source_sequence.dtype).min,
            )

        weights = torch.softmax(logits, dim=-1)
        values = scalar_sequence.transpose(1, 2).unsqueeze(2)
        pooled = torch.sum(weights * values, dim=-1)
        return pooled, weights


class T5EncoderBackbone(nn.Module):
    def __init__(self, shared_model: nn.Module) -> None:
        super().__init__()
        self.shared_model = shared_model
        self.model = shared_model.get_encoder()
        self.hidden_size = resolve_hidden_size_from_config(shared_model.config)

    def forward(
        self,
        input_ids: torch.LongTensor,
        attention_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True,
        )
        return outputs.last_hidden_state.float()


class T5DecoderBackbone(nn.Module):
    def __init__(self, shared_model: nn.Module) -> None:
        super().__init__()
        self.model = shared_model
        self.hidden_size = resolve_hidden_size_from_config(shared_model.config)

    @property
    def model_dtype(self) -> torch.dtype:
        first_param = next(self.model.parameters(), None)
        return torch.float32 if first_param is None else first_param.dtype

    @property
    def decoder_start_token_id(self) -> int:
        config = self.model.config
        start_token_id = config.decoder_start_token_id
        if start_token_id is None:
            start_token_id = getattr(config, "bos_token_id", None)
        if start_token_id is None:
            start_token_id = config.pad_token_id
        if start_token_id is None:
            raise ValueError("No decoder start token is configured.")
        return int(start_token_id)

    def seq2seq_forward(
        self,
        encoder_hidden_states: torch.Tensor,
        encoder_attention_mask: Optional[torch.Tensor],
        decoder_input_ids: Optional[torch.LongTensor] = None,
        decoder_attention_mask: Optional[torch.Tensor] = None,
        labels: Optional[torch.LongTensor] = None,
    ) -> Any:
        return self.model(
            attention_mask=encoder_attention_mask,
            encoder_outputs=BaseModelOutput(last_hidden_state=encoder_hidden_states.to(dtype=self.model_dtype)),
            decoder_input_ids=decoder_input_ids,
            decoder_attention_mask=decoder_attention_mask,
            labels=labels,
            use_cache=False,
            return_dict=True,
        )

    def generate_from_memory(
        self,
        encoder_hidden_states: torch.Tensor,
        encoder_attention_mask: Optional[torch.Tensor],
        decoder_input_ids: Optional[torch.LongTensor],
        max_new_tokens: int,
        num_beams: int = 1,
        do_sample: bool = False,
    ) -> torch.LongTensor:
        return self.model.generate(
            decoder_input_ids=decoder_input_ids,
            encoder_outputs=BaseModelOutput(last_hidden_state=encoder_hidden_states.to(dtype=self.model_dtype)),
            attention_mask=encoder_attention_mask,
            max_new_tokens=max_new_tokens,
            num_beams=num_beams,
            do_sample=do_sample,
        )


In [11]:

class FactorVAEEncoder(nn.Module):
    def __init__(
        self,
        input_dim: int,
        num_scalar_factors: int,
        vector_latent_dim: int,
        hidden_dim: int = 1024,
        dropout: float = 0.0,
    ) -> None:
        super().__init__()
        self.num_scalar_factors = num_scalar_factors
        self.vector_latent_dim = vector_latent_dim
        self.total_latent_dim = num_scalar_factors + vector_latent_dim

        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.mu = nn.Linear(hidden_dim, self.total_latent_dim)
        self.logvar = nn.Linear(hidden_dim, self.total_latent_dim)

    def split(self, tensor: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        scalar = tensor[..., : self.num_scalar_factors]
        vector = tensor[..., self.num_scalar_factors :]
        return scalar, vector

    def reparameterize(self, mu: torch.Tensor, logvar: torch.Tensor, sample: bool) -> torch.Tensor:
        if not sample:
            return mu
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x: torch.Tensor, sample: bool = True) -> FactorVAEOutput:
        hidden = self.net(x)
        mu = self.mu(hidden)
        logvar = self.logvar(hidden)
        z = self.reparameterize(mu=mu, logvar=logvar, sample=sample)

        scalar_z, vector_z = self.split(z)
        scalar_mu, vector_mu = self.split(mu)
        scalar_logvar, vector_logvar = self.split(logvar)
        return FactorVAEOutput(
            z=z,
            mu=mu,
            logvar=logvar,
            scalar_z=scalar_z,
            vector_z=vector_z,
            scalar_mu=scalar_mu,
            vector_mu=vector_mu,
            scalar_logvar=scalar_logvar,
            vector_logvar=vector_logvar,
        )


class FactorVAEDecoder(nn.Module):
    def __init__(
        self,
        output_dim: int,
        num_scalar_factors: int,
        vector_latent_dim: int,
        hidden_dim: int = 1024,
        dropout: float = 0.0,
    ) -> None:
        super().__init__()
        total_latent_dim = num_scalar_factors + vector_latent_dim
        self.net = nn.Sequential(
            nn.Linear(total_latent_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.net(z)


In [5]:

class T5FactorVAEModel(nn.Module):
    def __init__(
        self,
        model_config: ModelConfig,
        num_labels: int,
        pos_weight: Optional[torch.Tensor] = None,
    ) -> None:
        super().__init__()
        self.model_config = model_config
        self.num_scalar_factors = model_config.num_scalar_factors
        self.vector_latent_dim = model_config.vector_latent_dim
        self.num_labels = num_labels
        self.attention_source = model_config.attention_source
        self.pooling_mode = model_config.pooling_mode
        self.classifier_mode = model_config.classifier_mode
        self.classifier_uses_mu = model_config.classifier_uses_mu
        self.use_skip_connection = model_config.use_skip_connection

        valid_sources = {"scalar_only", "vector_only", "latent_full", "encoder_sequence"}
        if self.attention_source not in valid_sources:
            raise ValueError(f"Unsupported attention_source={self.attention_source}. Expected one of {sorted(valid_sources)}")
        valid_classifier_modes = {"joint_mlp", "per_emotion_mlp"}
        if self.classifier_mode not in valid_classifier_modes:
            raise ValueError(f"Unsupported classifier_mode={self.classifier_mode}. Expected one of {sorted(valid_classifier_modes)}")
        if self.classifier_mode == "per_emotion_mlp" and self.num_scalar_factors != num_labels:
            raise ValueError("per_emotion_mlp requires num_scalar_factors == num_labels")

        shared_t5 = AutoModelForSeq2SeqLM.from_pretrained(model_config.model_name)
        if model_config.use_lora:
            shared_t5 = get_peft_model(
                shared_t5,
                LoraConfig(
                    task_type=TaskType.SEQ_2_SEQ_LM,
                    r=model_config.lora_r,
                    lora_alpha=model_config.lora_alpha,
                    lora_dropout=model_config.lora_dropout,
                    target_modules=list(model_config.lora_target_modules),
                    bias="none",
                ),
            )
        self.shared_t5 = shared_t5
        self.t5_encoder = T5EncoderBackbone(shared_model=shared_t5)
        self.t5_decoder = T5DecoderBackbone(shared_model=shared_t5)
        self.hidden_size = self.t5_encoder.hidden_size

        self.vae_encoder = FactorVAEEncoder(
            input_dim=self.hidden_size,
            num_scalar_factors=self.num_scalar_factors,
            vector_latent_dim=self.vector_latent_dim,
            hidden_dim=model_config.vae_hidden_dim,
            dropout=model_config.vae_dropout,
        )

        source_dim_map = {
            "scalar_only": self.num_scalar_factors,
            "vector_only": self.vector_latent_dim,
            "latent_full": self.num_scalar_factors + self.vector_latent_dim,
            "encoder_sequence": self.hidden_size,
        }
        self.scalar_attention_pool = ScalarLatentAttentionPooling(
            num_scalar_factors=self.num_scalar_factors,
            source_dim=source_dim_map[self.attention_source],
            num_heads=model_config.latent_pool_heads,
            pooling_mode=model_config.pooling_mode,
            dropout=model_config.vae_dropout,
        )
        self.vae_decoder = FactorVAEDecoder(
            output_dim=self.hidden_size,
            num_scalar_factors=self.num_scalar_factors,
            vector_latent_dim=self.vector_latent_dim,
            hidden_dim=model_config.vae_hidden_dim,
            dropout=model_config.vae_dropout,
        )
        bottleneck_dim = max(1, min(int(model_config.residual_bottleneck_dim), self.hidden_size))
        self.residual_bottleneck = nn.Sequential(
            nn.Linear(self.hidden_size, bottleneck_dim),
            nn.GELU(),
            nn.Dropout(model_config.residual_bottleneck_dropout),
            nn.Linear(bottleneck_dim, self.hidden_size),
        )
        self.memory_norm = nn.LayerNorm(self.hidden_size)

        classifier_input_dim = self.num_scalar_factors * model_config.latent_pool_heads
        if self.classifier_mode == "joint_mlp":
            self.classifier = nn.Sequential(
                nn.Dropout(model_config.classifier_dropout),
                nn.Linear(classifier_input_dim, model_config.joint_mlp_hidden_dim),
                nn.Dropout(model_config.classifier_dropout),
                nn.Linear(model_config.joint_mlp_hidden_dim, model_config.joint_mlp_hidden_dim * 2),
                nn.Dropout(model_config.classifier_dropout),
                nn.Linear(model_config.joint_mlp_hidden_dim * 2, model_config.joint_mlp_hidden_dim),
                nn.Dropout(model_config.classifier_dropout),
                nn.Linear(model_config.joint_mlp_hidden_dim, num_labels),
            )
        else:
            self.emotion_classifiers = nn.ModuleList(
                [
                    nn.Sequential(
                        nn.Dropout(model_config.classifier_dropout),
                        nn.Linear(model_config.latent_pool_heads, model_config.per_emotion_hidden_dim),
                        nn.GELU(),
                        nn.Linear(model_config.per_emotion_hidden_dim, 1),
                    )
                    for _ in range(num_labels)
                ]
            )

        self.classification_loss_fn = nn.BCEWithLogitsLoss(
            pos_weight=None if pos_weight is None else pos_weight.float()
        )

        freeze_non_lora_parameters(self.shared_t5)
        self.shared_t5.eval()

    def train(self, mode: bool = True):
        super().train(mode)
        self.shared_t5.eval()
        return self

    def _attention_source_sequence(
        self,
        vae_out: FactorVAEOutput,
        t5_encoder_sequence: torch.Tensor,
    ) -> torch.Tensor:
        if self.classifier_uses_mu:
            if self.attention_source == "scalar_only":
                return vae_out.scalar_mu
            if self.attention_source == "vector_only":
                return vae_out.vector_mu
            if self.attention_source == "latent_full":
                return vae_out.mu
            return t5_encoder_sequence

        if self.attention_source == "scalar_only":
            return vae_out.scalar_z
        if self.attention_source == "vector_only":
            return vae_out.vector_z
        if self.attention_source == "latent_full":
            return vae_out.z
        return t5_encoder_sequence

    def encode(
        self,
        input_ids: torch.LongTensor,
        attention_mask: Optional[torch.Tensor],
        sample_posterior: bool,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, FactorVAEOutput]:
        t5_encoder_sequence = self.t5_encoder(input_ids=input_ids, attention_mask=attention_mask)
        vae_out = self.vae_encoder(t5_encoder_sequence, sample=sample_posterior)
        source_sequence = self._attention_source_sequence(vae_out=vae_out, t5_encoder_sequence=t5_encoder_sequence)
        scalar_sequence = vae_out.scalar_mu if self.classifier_uses_mu else vae_out.scalar_z
        pooled_scalar_tensor, attn_weights = self.scalar_attention_pool(
            source_sequence=source_sequence,
            scalar_sequence=scalar_sequence,
            attention_mask=attention_mask,
        )
        return t5_encoder_sequence, attn_weights, pooled_scalar_tensor, vae_out

    def decode(
        self,
        vae_out: FactorVAEOutput,
        t5_encoder_sequence: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        vae_decoded_sequence = self.vae_decoder(z=vae_out.z)
        residual_memory = self.residual_bottleneck(t5_encoder_sequence)
        decoder_memory = self.memory_norm(vae_decoded_sequence + residual_memory)
        return vae_decoded_sequence, decoder_memory

    def classification_logits(self, pooled_scalar_tensor: torch.Tensor) -> torch.Tensor:
        if self.classifier_mode == "joint_mlp":
            features = pooled_scalar_tensor.flatten(start_dim=1)
            return self.classifier(features)
        logits = [
            head(pooled_scalar_tensor[:, idx, :])
            for idx, head in enumerate(self.emotion_classifiers)
        ]
        return torch.cat(logits, dim=-1)

    def base_losses(
        self,
        classification_logits: torch.Tensor,
        t5_encoder_sequence: torch.Tensor,
        vae_decoded_sequence: torch.Tensor,
        vae_out: FactorVAEOutput,
        attention_mask: Optional[torch.Tensor],
        labels: Optional[torch.Tensor],
        classification_weight: float,
        recon_weight: float,
        kl_weight: float,
    ) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        if attention_mask is None:
            recon_loss = F.mse_loss(vae_decoded_sequence, t5_encoder_sequence)
        else:
            mask = attention_mask.to(t5_encoder_sequence.dtype).unsqueeze(-1)
            sq_error = (vae_decoded_sequence - t5_encoder_sequence).pow(2)
            denom = mask.sum().clamp_min(1.0) * sq_error.size(-1)
            recon_loss = (sq_error * mask).sum() / denom

        kl_per_token = -0.5 * (1 + vae_out.logvar - vae_out.mu.pow(2) - vae_out.logvar.exp()).sum(dim=-1)
        if attention_mask is None:
            kl_loss = kl_per_token.mean()
        else:
            mask = attention_mask.to(kl_per_token.dtype)
            kl_loss = (kl_per_token * mask).sum() / mask.sum().clamp_min(1.0)

        total = (recon_weight * recon_loss) + (kl_weight * kl_loss)
        loss_terms = {
            "reconstruction": recon_loss.detach(),
            "kl": kl_loss.detach(),
        }
        if labels is not None:
            classification_loss = self.classification_loss_fn(classification_logits, labels.float())
            total = total + (classification_weight * classification_loss)
            loss_terms["classification"] = classification_loss.detach()
        return total, loss_terms

    def forward(
        self,
        input_ids: torch.LongTensor,
        attention_mask: Optional[torch.Tensor],
        labels: Optional[torch.Tensor],
        sample_posterior: bool,
        classification_weight: float,
        recon_weight: float,
        kl_weight: float,
    ) -> T5FactorVAEModelOutput:
        t5_encoder_sequence, attn_weights, pooled_scalar_tensor, vae_out = self.encode(
            input_ids=input_ids,
            attention_mask=attention_mask,
            sample_posterior=sample_posterior,
        )
        vae_decoded_sequence, decoder_memory = self.decode(
            vae_out=vae_out,
            t5_encoder_sequence=t5_encoder_sequence,
        )
        classification_logits = self.classification_logits(pooled_scalar_tensor)

        base_loss = None
        loss_terms = None
        if labels is not None:
            base_loss, loss_terms = self.base_losses(
                classification_logits=classification_logits,
                t5_encoder_sequence=t5_encoder_sequence,
                vae_decoded_sequence=vae_decoded_sequence,
                vae_out=vae_out,
                attention_mask=attention_mask,
                labels=labels,
                classification_weight=classification_weight,
                recon_weight=recon_weight,
                kl_weight=kl_weight,
            )

        return T5FactorVAEModelOutput(
            t5_encoder_sequence=t5_encoder_sequence,
            attention_weights=attn_weights,
            pooled_scalar_tensor=pooled_scalar_tensor,
            vae=vae_out,
            vae_decoded_sequence=vae_decoded_sequence,
            decoder_memory=decoder_memory,
            classification_logits=classification_logits,
            base_loss=base_loss,
            loss_terms=loss_terms,
        )


## FactorVAE total correlation and copy helpers

Two key fixes live here.

### 1. Total correlation

The original notebook flattened token latents across the batch and applied the discriminator there. That does **not** match the usual FactorVAE objective. Here the discriminator sees a **per-example latent summary** obtained by masked mean pooling over valid token latents.

That means:

- the VAE still remains **token-level**,
- the TC penalty is applied at the **example level**,
- the scalar-vs-vector semantic split is still mainly driven by the **emotion head attached to the scalar factors**, while TC encourages a more factorized aggregate posterior.

### 2. Copy training

The helper below builds decoder teacher-forcing inputs in a way that is consistent with the selected prompt policy.


In [13]:
class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x: torch.Tensor, weight: float):
        ctx.weight = float(weight)
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output: torch.Tensor):
        return (-ctx.weight) * grad_output, None



def grad_reverse(x: torch.Tensor, weight: float = 1.0) -> torch.Tensor:
    return GradientReversalFunction.apply(x, weight)


class FactorVAEDiscriminator(nn.Module):
    def __init__(self, latent_dim: int, hidden_dim: int = 512) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, 2),
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.net(z)


class VectorEmotionAdversary(nn.Module):
    def __init__(self, vector_dim: int, num_labels: int, hidden_dim: int = 128) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(vector_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, num_labels),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)



def permute_latent_dims(z: torch.Tensor) -> torch.Tensor:
    columns = []
    for dim_idx in range(z.size(1)):
        permutation = torch.randperm(z.size(0), device=z.device)
        columns.append(z[permutation, dim_idx])
    return torch.stack(columns, dim=1)



def tc_subspace_dim(loss_config: LossConfig, model_config: ModelConfig) -> int:
    if loss_config.tc_subspace == "none":
        return 0
    if loss_config.tc_subspace == "scalar":
        return int(model_config.num_scalar_factors)
    if loss_config.tc_subspace == "vector":
        return int(model_config.vector_latent_dim)
    if loss_config.tc_subspace == "full":
        return int(model_config.num_scalar_factors + model_config.vector_latent_dim)
    raise ValueError(f"Unsupported tc_subspace={loss_config.tc_subspace}")



def pooled_latents_for_tc(
    vae_out: FactorVAEOutput,
    attention_mask: Optional[torch.Tensor],
    tc_subspace: str,
) -> Optional[torch.Tensor]:
    if tc_subspace == "none":
        return None
    if tc_subspace == "scalar":
        return masked_mean(vae_out.scalar_z, attention_mask)
    if tc_subspace == "vector":
        return masked_mean(vae_out.vector_z, attention_mask)
    if tc_subspace == "full":
        return masked_mean(vae_out.z, attention_mask)
    raise ValueError(f"Unsupported tc_subspace={tc_subspace}")



def factorvae_tc_loss(discriminator: nn.Module, z_batch: torch.Tensor) -> torch.Tensor:
    if z_batch is None or z_batch.size(0) < 2:
        if z_batch is None:
            return torch.zeros((), device=device)
        return z_batch.new_zeros(())
    logits = discriminator(z_batch)
    return (logits[:, 0] - logits[:, 1]).mean()



def discriminator_loss(discriminator: nn.Module, z_batch: torch.Tensor) -> torch.Tensor:
    if z_batch is None or z_batch.size(0) < 2:
        if z_batch is None:
            return torch.zeros((), device=device)
        return z_batch.new_zeros(())
    z_detached = z_batch.detach()
    z_perm = permute_latent_dims(z_detached)
    real_logits = discriminator(z_detached)
    perm_logits = discriminator(z_perm)
    real_target = torch.zeros(z_detached.size(0), dtype=torch.long, device=z_detached.device)
    perm_target = torch.ones(z_detached.size(0), dtype=torch.long, device=z_detached.device)
    ce = nn.CrossEntropyLoss()
    return 0.5 * (ce(real_logits, real_target) + ce(perm_logits, perm_target))



def multilabel_bce_loss(
    logits: torch.Tensor,
    labels: torch.Tensor,
    pos_weight: Optional[torch.Tensor] = None,
) -> torch.Tensor:
    if pos_weight is not None:
        pos_weight = pos_weight.to(device=logits.device, dtype=logits.dtype)
    return F.binary_cross_entropy_with_logits(logits, labels.float(), pos_weight=pos_weight)



def vector_adversarial_loss(
    vector_adversary: nn.Module,
    vector_summary: torch.Tensor,
    labels: torch.Tensor,
    pos_weight: Optional[torch.Tensor],
    grl_weight: float = 1.0,
) -> Tuple[torch.Tensor, torch.Tensor]:
    logits = vector_adversary(grad_reverse(vector_summary, grl_weight))
    loss = multilabel_bce_loss(logits, labels, pos_weight=pos_weight)
    return logits, loss



def vector_probe_loss(
    vector_adversary: nn.Module,
    vector_summary: torch.Tensor,
    labels: torch.Tensor,
    pos_weight: Optional[torch.Tensor],
) -> Tuple[torch.Tensor, torch.Tensor]:
    logits = vector_adversary(vector_summary)
    loss = multilabel_bce_loss(logits, labels, pos_weight=pos_weight)
    return logits, loss



def cross_covariance_penalty(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    if a.size(0) <= 1 or b.size(0) <= 1:
        return a.new_zeros(())
    a_centered = a - a.mean(dim=0, keepdim=True)
    b_centered = b - b.mean(dim=0, keepdim=True)
    cov = a_centered.transpose(0, 1) @ b_centered
    cov = cov / float(max(a.size(0) - 1, 1))
    return cov.pow(2).mean()


In [14]:
def prompt_token_ids(tokenizer: AutoTokenizer, prompt_text: str) -> List[int]:
    if not prompt_text:
        return []
    return tokenizer(prompt_text, add_special_tokens=False)["input_ids"]



def build_copy_teacher_forcing_batch(
    texts: Sequence[str],
    tokenizer: AutoTokenizer,
    decoder_start_token_id: int,
    prompt_config: PromptConfig,
    device: torch.device,
) -> Dict[str, torch.Tensor]:
    pad_token_id = tokenizer.pad_token_id
    if pad_token_id is None:
        raise ValueError("Tokenizer must define pad_token_id.")

    prompt_ids = prompt_token_ids(tokenizer, prompt_config.prompt_text) if prompt_config.use_prompt else []
    eos_id = tokenizer.eos_token_id
    max_len = prompt_config.max_decoder_length

    teacher_force = torch.full((len(texts), max_len), pad_token_id, dtype=torch.long)
    labels = torch.full((len(texts), max_len), -100, dtype=torch.long)
    decoder_attention_mask = torch.zeros((len(texts), max_len), dtype=torch.long)

    for row_idx, text in enumerate(texts):
        target_ids = tokenizer(text, add_special_tokens=False)["input_ids"]
        full_ids = list(prompt_ids) + list(target_ids)
        if eos_id is not None:
            full_ids = full_ids + [eos_id]
        full_ids = full_ids[:max_len]
        seq_len = len(full_ids)
        if seq_len == 0:
            continue

        teacher_force[row_idx, :seq_len] = torch.tensor(full_ids, dtype=torch.long)
        labels[row_idx, :seq_len] = torch.tensor(full_ids, dtype=torch.long)
        decoder_attention_mask[row_idx, :seq_len] = 1

        if prompt_config.use_prompt and prompt_config.mask_prompt_loss:
            prompt_len = min(len(prompt_ids), seq_len)
            if prompt_len > 0:
                labels[row_idx, :prompt_len] = -100

    decoder_input_ids = shift_tokens_right_manual(
        input_ids=teacher_force,
        pad_token_id=pad_token_id,
        decoder_start_token_id=decoder_start_token_id,
    )

    return {
        "decoder_input_ids": decoder_input_ids.to(device),
        "decoder_attention_mask": decoder_attention_mask.to(device),
        "labels": labels.to(device),
    }



def build_generation_prefix(
    tokenizer: AutoTokenizer,
    batch_size: int,
    prompt_config: PromptConfig,
    device: torch.device,
) -> Tuple[Optional[torch.LongTensor], int]:
    if not prompt_config.use_prompt:
        return None, 0
    ids = tokenizer(prompt_config.prompt_text, add_special_tokens=False, return_tensors="pt")["input_ids"].to(device)
    return ids.repeat(batch_size, 1), int(ids.size(1))



def decode_generated_batch(
    tokenizer: AutoTokenizer,
    generated_ids: torch.LongTensor,
    prompt_prefix_len: int,
) -> List[str]:
    cpu_ids = generated_ids.detach().cpu()
    if prompt_prefix_len > 0:
        cpu_ids = cpu_ids[:, prompt_prefix_len:]
    return tokenizer.batch_decode(cpu_ids, skip_special_tokens=True)



def compute_copy_loss_from_memory(
    model: T5FactorVAEModel,
    tokenizer: AutoTokenizer,
    decoder_memory: torch.Tensor,
    encoder_attention_mask: torch.Tensor,
    texts: Sequence[str],
    prompt_config: PromptConfig,
    device: torch.device,
) -> torch.Tensor:
    batch = build_copy_teacher_forcing_batch(
        texts=texts,
        tokenizer=tokenizer,
        decoder_start_token_id=model.t5_decoder.decoder_start_token_id,
        prompt_config=prompt_config,
        device=device,
    )
    outputs = model.t5_decoder.seq2seq_forward(
        encoder_hidden_states=decoder_memory,
        encoder_attention_mask=encoder_attention_mask,
        decoder_input_ids=batch["decoder_input_ids"],
        decoder_attention_mask=batch["decoder_attention_mask"],
        labels=batch["labels"],
    )
    if outputs.loss is None:
        raise RuntimeError("Expected copy loss, but model returned None.")
    return outputs.loss



def generate_text_from_memory(
    model: T5FactorVAEModel,
    tokenizer: AutoTokenizer,
    decoder_memory: torch.Tensor,
    encoder_attention_mask: torch.Tensor,
    prompt_config: PromptConfig,
    max_new_tokens: int,
    num_beams: int = 1,
    do_sample: bool = False,
) -> List[str]:
    decoder_prefix, prompt_prefix_len = build_generation_prefix(
        tokenizer=tokenizer,
        batch_size=decoder_memory.size(0),
        prompt_config=prompt_config,
        device=decoder_memory.device,
    )
    generated_ids = model.t5_decoder.generate_from_memory(
        encoder_hidden_states=decoder_memory,
        encoder_attention_mask=encoder_attention_mask,
        decoder_input_ids=decoder_prefix,
        max_new_tokens=max_new_tokens,
        num_beams=num_beams,
        do_sample=do_sample,
    )
    return decode_generated_batch(
        tokenizer=tokenizer,
        generated_ids=generated_ids,
        prompt_prefix_len=prompt_prefix_len,
    )


## Text-copy metrics and multilabel evaluation

The original notebook relied on a weak string heuristic for prompt comparison and mixed several thresholding conventions.

This version uses:

- a single **global threshold** tuned on a held-out train-side calibration split for **micro-F1**,
- consistent reuse of that threshold everywhere,
- stronger text-copy metrics:
  - exact match,
  - token F1,
  - normalized character edit similarity.


### Note on raw loss values

The raw KL term can easily look much larger than the reconstruction or classification loss because it is summed across latent dimensions before averaging over valid tokens.

That does **not** mean it dominates optimization. What matters for training is the **weighted contribution** of each term. The training printouts below therefore show both:

- raw losses,
- weighted contributions used in the objective.


In [15]:
def normalize_text_for_copy_metric(text: str) -> str:
    return " ".join(str(text).strip().split())



def token_f1_score(reference_text: str, generated_text: str) -> float:
    ref_tokens = normalize_text_for_copy_metric(reference_text).lower().split()
    gen_tokens = normalize_text_for_copy_metric(generated_text).lower().split()
    if not ref_tokens and not gen_tokens:
        return 1.0
    if not ref_tokens or not gen_tokens:
        return 0.0
    ref_counter = Counter(ref_tokens)
    gen_counter = Counter(gen_tokens)
    overlap = sum((ref_counter & gen_counter).values())
    precision = overlap / max(len(gen_tokens), 1)
    recall = overlap / max(len(ref_tokens), 1)
    if (precision + recall) <= 1e-12:
        return 0.0
    return float(2.0 * precision * recall / (precision + recall))



def summarize_copy_metrics(reference_texts: Sequence[str], generated_texts: Sequence[str]) -> Dict[str, float]:
    if len(reference_texts) != len(generated_texts):
        raise ValueError("reference_texts and generated_texts must have the same length.")
    if len(reference_texts) == 0:
        return {"exact_match": 0.0, "token_f1": 0.0, "edit_similarity": 0.0}

    exact = []
    token_f1 = []
    edit_similarity = []
    for reference, generated in zip(reference_texts, generated_texts):
        ref_norm = normalize_text_for_copy_metric(reference)
        gen_norm = normalize_text_for_copy_metric(generated)
        exact.append(1.0 if ref_norm == gen_norm else 0.0)
        token_f1.append(token_f1_score(ref_norm, gen_norm))
        edit_similarity.append(SequenceMatcher(a=ref_norm, b=gen_norm).ratio())

    return {
        "exact_match": float(np.mean(exact)),
        "token_f1": float(np.mean(token_f1)),
        "edit_similarity": float(np.mean(edit_similarity)),
    }



def tune_global_threshold_for_micro_f1(
    logits: torch.Tensor,
    labels: torch.Tensor,
    num_points: int = 101,
) -> float:
    probs = torch.sigmoid(logits.detach().cpu()).numpy()
    targets = labels.detach().cpu().numpy().astype(int)

    best_threshold = 0.5
    best_score = -1.0
    for threshold in np.linspace(0.01, 0.99, num_points):
        preds = (probs >= threshold).astype(int)
        score = f1_score(targets, preds, average="micro", zero_division=0)
        if score > best_score:
            best_score = float(score)
            best_threshold = float(threshold)
    return best_threshold



def multilabel_metrics(
    logits: torch.Tensor,
    labels: torch.Tensor,
    threshold: float,
    label_names: Sequence[str],
) -> Dict[str, Any]:
    probs_tensor = torch.sigmoid(logits.detach().cpu())
    labels_tensor = labels.detach().cpu()
    probs = probs_tensor.numpy()
    targets = labels_tensor.numpy().astype(int)
    preds = (probs >= float(threshold)).astype(int)

    def safe_average_precision_micro(y_true: np.ndarray, y_score: np.ndarray) -> float:
        try:
            value = average_precision_score(y_true, y_score, average="micro")
            return float(value) if np.isfinite(value) else 0.0
        except ValueError:
            return 0.0

    def safe_lrap(y_true: np.ndarray, y_score: np.ndarray) -> float:
        try:
            value = label_ranking_average_precision_score(y_true, y_score)
            return float(value) if np.isfinite(value) else 0.0
        except ValueError:
            return 0.0

    return {
        "threshold": float(threshold),
        "micro_f1": float(f1_score(targets, preds, average="micro", zero_division=0)),
        "macro_f1": float(f1_score(targets, preds, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(targets, preds, average="weighted", zero_division=0)),
        "subset_accuracy": float(accuracy_score(targets, preds)),
        "hamming_acc": float(1.0 - hamming_loss(targets, preds)),
        "jaccard_micro": float(jaccard_score(targets, preds, average="micro", zero_division=0)),
        "average_precision_micro": safe_average_precision_micro(targets, probs),
        "label_ranking_average_precision": safe_lrap(targets, probs),
        "classification_report_text": classification_report(
            targets,
            preds,
            target_names=list(label_names),
            zero_division=0,
        ),
        "logits": logits.detach().cpu(),
        "labels": labels_tensor,
        "probs": probs_tensor,
        "preds": torch.from_numpy(preds),
    }



def evaluate_loader(
    state: TrainingState,
    ctx: ExperimentContext,
    data_loader: DataLoader,
    desc: str,
    tune_threshold: bool,
    weights: Dict[str, float],
) -> Dict[str, Any]:
    model = state.model
    discriminator = state.discriminator
    vector_adversary = state.vector_adversary
    model.eval()
    if vector_adversary is not None:
        vector_adversary.eval()

    total_base_loss = 0.0
    total_total_loss = 0.0
    total_recon = 0.0
    total_kl = 0.0
    total_cls = 0.0
    total_tc = 0.0
    total_copy = 0.0
    total_vec_adv = 0.0
    total_sep = 0.0
    steps = 0

    all_logits = []
    all_labels = []

    with torch.inference_mode():
        for batch in tqdm(data_loader, desc=desc, leave=False):
            input_ids = batch["input_ids"].to(ctx.device)
            attention_mask = batch["attention_mask"].to(ctx.device)
            labels = batch["labels"].to(ctx.device)
            texts = batch["texts"]

            out = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
                sample_posterior=ctx.experiment_config.sample_posterior_eval,
                classification_weight=weights["classification_weight"],
                recon_weight=weights["recon_weight"],
                kl_weight=weights["kl_weight"],
            )

            tc_inputs = pooled_latents_for_tc(out.vae, attention_mask, ctx.loss_config.tc_subspace)
            tc_loss = out.base_loss.new_zeros(()) if discriminator is None else factorvae_tc_loss(discriminator, tc_inputs)

            copy_loss = out.base_loss.new_zeros(())
            if weights["copy_weight"] > 0.0:
                copy_loss = compute_copy_loss_from_memory(
                    model=model,
                    tokenizer=ctx.tokenizer,
                    decoder_memory=out.decoder_memory,
                    encoder_attention_mask=attention_mask,
                    texts=texts,
                    prompt_config=ctx.prompt_config,
                    device=ctx.device,
                )

            vector_summary = masked_mean(out.vae.vector_mu, attention_mask)
            scalar_summary = masked_mean(out.vae.scalar_mu, attention_mask)
            vec_adv_loss = out.base_loss.new_zeros(())
            if vector_adversary is not None:
                _, vec_adv_loss = vector_probe_loss(
                    vector_adversary=vector_adversary,
                    vector_summary=vector_summary,
                    labels=labels,
                    pos_weight=ctx.pos_weight,
                )
            sep_loss = cross_covariance_penalty(scalar_summary, vector_summary)

            total_loss = (
                out.base_loss
                + (weights["tc_weight"] * tc_loss)
                + (weights["copy_weight"] * copy_loss)
                + (weights["vector_adv_weight"] * vec_adv_loss)
                + (weights["vector_sep_weight"] * sep_loss)
            )

            all_logits.append(out.classification_logits.detach().cpu())
            all_labels.append(labels.detach().cpu())
            total_base_loss += float(out.base_loss.detach().cpu().item())
            total_total_loss += float(total_loss.detach().cpu().item())
            total_recon += float(out.loss_terms["reconstruction"].item())
            total_kl += float(out.loss_terms["kl"].item())
            total_cls += float(out.loss_terms["classification"].item())
            total_tc += float(tc_loss.detach().cpu().item())
            total_copy += float(copy_loss.detach().cpu().item())
            total_vec_adv += float(vec_adv_loss.detach().cpu().item())
            total_sep += float(sep_loss.detach().cpu().item())
            steps += 1

    logits = torch.cat(all_logits, dim=0)
    labels = torch.cat(all_labels, dim=0)
    threshold = tune_global_threshold_for_micro_f1(
        logits=logits,
        labels=labels,
        num_points=ctx.experiment_config.threshold_grid_points,
    ) if tune_threshold else state.threshold

    metrics = multilabel_metrics(
        logits=logits,
        labels=labels,
        threshold=threshold,
        label_names=ctx.emotion_names,
    )
    metrics.update(
        {
            "avg_base_loss": total_base_loss / max(steps, 1),
            "avg_total_loss": total_total_loss / max(steps, 1),
            "avg_recon": total_recon / max(steps, 1),
            "avg_kl": total_kl / max(steps, 1),
            "avg_cls": total_cls / max(steps, 1),
            "avg_tc": total_tc / max(steps, 1),
            "avg_copy": total_copy / max(steps, 1),
            "avg_vec_adv": total_vec_adv / max(steps, 1),
            "avg_sep": total_sep / max(steps, 1),
            "weighted_recon": weights["recon_weight"] * (total_recon / max(steps, 1)),
            "weighted_kl": weights["kl_weight"] * (total_kl / max(steps, 1)),
            "weighted_cls": weights["classification_weight"] * (total_cls / max(steps, 1)),
            "weighted_tc": weights["tc_weight"] * (total_tc / max(steps, 1)),
            "weighted_copy": weights["copy_weight"] * (total_copy / max(steps, 1)),
            "weighted_vec_adv": weights["vector_adv_weight"] * (total_vec_adv / max(steps, 1)),
            "weighted_sep": weights["vector_sep_weight"] * (total_sep / max(steps, 1)),
            "steps": steps,
        }
    )
    release_memory()
    return metrics



def summarize_eval_metrics(prefix: str, metrics: Dict[str, Any]) -> None:
    print(
        f"{prefix}loss={metrics['avg_total_loss']:.4f} "
        f"raw(recon={metrics['avg_recon']:.4f}, kl={metrics['avg_kl']:.4f}, cls={metrics['avg_cls']:.4f}, tc={metrics['avg_tc']:.4f}, copy={metrics['avg_copy']:.4f}, vec_adv={metrics['avg_vec_adv']:.4f}, sep={metrics['avg_sep']:.4f}) "
        f"weighted(recon={metrics['weighted_recon']:.4f}, kl={metrics['weighted_kl']:.4f}, cls={metrics['weighted_cls']:.4f}, tc={metrics['weighted_tc']:.4f}, copy={metrics['weighted_copy']:.4f}, vec_adv={metrics['weighted_vec_adv']:.4f}, sep={metrics['weighted_sep']:.4f}) "
        f"threshold={metrics['threshold']:.3f} "
        f"micro_f1={metrics['micro_f1']:.4f} "
        f"macro_f1={metrics['macro_f1']:.4f} "
        f"weighted_f1={metrics['weighted_f1']:.4f} "
        f"subset_acc={metrics['subset_accuracy']:.4f} "
        f"hamming_acc={metrics['hamming_acc']:.4f} "
        f"jaccard_micro={metrics['jaccard_micro']:.4f} "
        f"ap_micro={metrics['average_precision_micro']:.4f} "
        f"lrap={metrics['label_ranking_average_precision']:.4f}"
    )


## Instantiate the experiment

The defaults below preserve the architecture while fixing the problems:

- **bottlenecked latent**: `28 + 128 = 156 < 512` for T5-small,
- **no skip connection**,
- **frozen T5** with **LoRA only**,
- **scalar-only** attention source for the main run,
- **per-emotion** classifier for the main run.

The ablation options are still available by changing config values.


In [16]:
model = T5FactorVAEModel(
    model_config=MODEL_CONFIG,
    num_labels=num_labels,
    pos_weight=pos_weight,
).to(device)

tc_latent_dim = tc_subspace_dim(LOSS_CONFIG, MODEL_CONFIG)
discriminator = None
if LOSS_CONFIG.tc_weight > 0.0 and LOSS_CONFIG.tc_subspace != "none":
    discriminator = FactorVAEDiscriminator(
        latent_dim=tc_latent_dim,
        hidden_dim=512,
    ).to(device)

vector_adversary = None
if LOSS_CONFIG.vector_adv_weight > 0.0:
    vector_adversary = VectorEmotionAdversary(
        vector_dim=MODEL_CONFIG.vector_latent_dim,
        num_labels=num_labels,
        hidden_dim=MODEL_CONFIG.vector_adversary_hidden_dim,
    ).to(device)

# LoRA starts disabled; the schedule selectively enables decoder cross-attention adapters.
set_lora_trainable(model.shared_t5, enabled=False)

optimizer = torch.optim.AdamW(
    optimizable_model_parameters(model),
    lr=SCHEDULE_CONFIG.lr,
    weight_decay=SCHEDULE_CONFIG.weight_decay,
)
disc_optimizer = None if discriminator is None else torch.optim.AdamW(
    discriminator.parameters(),
    lr=SCHEDULE_CONFIG.disc_lr,
    weight_decay=SCHEDULE_CONFIG.weight_decay,
)
vector_adv_optimizer = None if vector_adversary is None else torch.optim.AdamW(
    vector_adversary.parameters(),
    lr=SCHEDULE_CONFIG.vector_adv_lr,
    weight_decay=SCHEDULE_CONFIG.weight_decay,
)

scheduler_total_steps = max(1, SCHEDULE_CONFIG.num_epochs * len(train_loader))
scheduler_warmup_steps = max(1, int(SCHEDULE_CONFIG.warmup_ratio * scheduler_total_steps))


def lr_lambda(current_step: int) -> float:
    if current_step < scheduler_warmup_steps:
        return float(current_step + 1) / float(max(1, scheduler_warmup_steps))
    progress = float(current_step - scheduler_warmup_steps) / float(max(1, scheduler_total_steps - scheduler_warmup_steps))
    progress = min(max(progress, 0.0), 1.0)
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return SCHEDULE_CONFIG.min_lr_scale + (1.0 - SCHEDULE_CONFIG.min_lr_scale) * cosine


lr_scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)

ctx = ExperimentContext(
    tokenizer=tokenizer,
    emotion_names=emotion_names,
    device=device,
    data_config=DATA_CONFIG,
    prompt_config=PROMPT_CONFIG,
    model_config=MODEL_CONFIG,
    loss_config=LOSS_CONFIG,
    schedule_config=SCHEDULE_CONFIG,
    experiment_config=EXPERIMENT_CONFIG,
    pos_weight=pos_weight.to(device),
)

state = TrainingState(
    model=model,
    discriminator=discriminator,
    vector_adversary=vector_adversary,
    optimizer=optimizer,
    disc_optimizer=disc_optimizer,
    vector_adv_optimizer=vector_adv_optimizer,
    lr_scheduler=lr_scheduler,
    threshold=0.5,
    best_val_micro_f1=float("-inf"),
    step=0,
    history=[],
)

output_dir = Path(EXPERIMENT_CONFIG.output_dir)
output_dir.mkdir(parents=True, exist_ok=True)
save_json(
    output_dir / "config.json",
    {
        "data_config": asdict(DATA_CONFIG),
        "prompt_config": asdict(PROMPT_CONFIG),
        "model_config": asdict(MODEL_CONFIG),
        "loss_config": asdict(LOSS_CONFIG),
        "schedule_config": asdict(SCHEDULE_CONFIG),
        "experiment_config": asdict(EXPERIMENT_CONFIG),
        "emotion_names": list(emotion_names),
    },
)

trainable_names = [name for name, param in model.named_parameters() if param.requires_grad]
print("hidden size:", model.hidden_size)
print("latent dim:", MODEL_CONFIG.num_scalar_factors + MODEL_CONFIG.vector_latent_dim)
print("trainable parameter groups:", len(trainable_names))
print("first ten trainable names:", trainable_names[:10])
if vector_adversary is not None:
    adv_params = sum(param.numel() for param in vector_adversary.parameters())
    print("vector adversary params:", adv_params)
print("output_dir:", output_dir.resolve())

if torch.cuda.is_available():
    total_gb = torch.cuda.get_device_properties(device).total_memory / (1024 ** 3)
    print(f"gpu total memory: {total_gb:.2f} GB")
    print(f"configured batch sizes -> train={DATA_CONFIG.train_batch_size}, eval={DATA_CONFIG.eval_batch_size}")


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


hidden size: 512
latent dim: 416
trainable parameter groups: 30
first ten trainable names: ['vae_encoder.net.0.weight', 'vae_encoder.net.0.bias', 'vae_encoder.net.3.weight', 'vae_encoder.net.3.bias', 'vae_encoder.net.6.weight', 'vae_encoder.net.6.bias', 'vae_encoder.mu.weight', 'vae_encoder.mu.bias', 'vae_encoder.logvar.weight', 'vae_encoder.logvar.bias']
output_dir: /mnt/disk1/Projects/NLP-Latent-Learning/factorvae_tokenlevel_clean_artifacts
gpu total memory: 3.68 GB
configured batch sizes -> train=8, eval=8


## Prompt comparison helper

This experiment is optional and belongs **after training**. It evaluates prompt candidates using real copy-quality metrics instead of a single heuristic string ratio.


In [17]:

def evaluate_prompt_candidates(
    state: TrainingState,
    ctx: ExperimentContext,
    texts: Sequence[str],
    prompt_candidates: Sequence[str],
    use_vae_memory: bool = True,
    max_new_tokens: int = 128,
) -> List[Dict[str, Any]]:
    model = state.model
    model.eval()

    results: List[Dict[str, Any]] = []
    for prompt_text in prompt_candidates:
        candidate_prompt_cfg = PromptConfig(
            use_prompt=bool(prompt_text),
            prompt_text=prompt_text,
            mask_prompt_loss=ctx.prompt_config.mask_prompt_loss,
            max_decoder_length=ctx.prompt_config.max_decoder_length,
        )
        generated_texts: List[str] = []

        for start_idx in range(0, len(texts), ctx.data_config.eval_batch_size):
            batch_texts = list(texts[start_idx : start_idx + ctx.data_config.eval_batch_size])
            encoded = ctx.tokenizer(
                batch_texts,
                max_length=ctx.data_config.max_length,
                padding=True,
                truncation=True,
                return_tensors="pt",
            )
            input_ids = encoded["input_ids"].to(ctx.device)
            attention_mask = encoded["attention_mask"].to(ctx.device)

            with torch.no_grad():
                out = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=None,
                    sample_posterior=ctx.experiment_config.sample_posterior_eval,
                    classification_weight=ctx.loss_config.classification_weight,
                    recon_weight=ctx.loss_config.recon_weight,
                    kl_weight=ctx.loss_config.kl_weight,
                )
                memory = out.decoder_memory if use_vae_memory else out.t5_encoder_sequence
                batch_generated = generate_text_from_memory(
                    model=model,
                    tokenizer=ctx.tokenizer,
                    decoder_memory=memory,
                    encoder_attention_mask=attention_mask,
                    prompt_config=candidate_prompt_cfg,
                    max_new_tokens=max_new_tokens,
                    num_beams=1,
                    do_sample=False,
                )
                generated_texts.extend(batch_generated)

        metrics = summarize_copy_metrics(texts, generated_texts)
        results.append(
            {
                "prompt": prompt_text,
                "use_vae_memory": use_vae_memory,
                **metrics,
            }
        )

    return sorted(results, key=lambda row: (row["exact_match"], row["token_f1"], row["edit_similarity"]), reverse=True)


## Qualitative monitoring helpers

The next block defines the helper functions used during training.

These helpers make the per-epoch output readable by printing the same compact report every epoch on two fixed validation examples.


In [18]:
def make_partial_desired_emotion_target(
    label_names: Sequence[str],
    positive_labels: Sequence[str],
) -> Tuple[torch.Tensor, torch.Tensor]:
    target = torch.zeros(len(label_names), dtype=torch.float32)
    mask = torch.zeros(len(label_names), dtype=torch.float32)
    name_to_idx = {name: idx for idx, name in enumerate(label_names)}
    for name in positive_labels:
        if name not in name_to_idx:
            raise ValueError(f"Unknown emotion label: {name}")
        idx = name_to_idx[name]
        target[idx] = 1.0
        mask[idx] = 1.0
    return target, mask



def masked_bce_with_logits(logits: torch.Tensor, targets: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    raw = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    denom = mask.sum().clamp_min(1.0)
    return (raw * mask).sum() / denom



def optimize_emotion_latents_for_target(
    state: TrainingState,
    ctx: ExperimentContext,
    input_ids: torch.LongTensor,
    attention_mask: torch.Tensor,
    desired_target: torch.Tensor,
    desired_mask: torch.Tensor,
    steps: int = 80,
    lr: float = 3e-2,
    emotion_l2_weight: float = 5e-3,
) -> Dict[str, Any]:
    model = state.model
    model.eval()

    input_ids = input_ids.to(ctx.device)
    attention_mask = attention_mask.to(ctx.device)
    desired_target = desired_target.to(ctx.device).float().unsqueeze(0)
    desired_mask = desired_mask.to(ctx.device).float().unsqueeze(0)

    with torch.no_grad():
        t5_encoder_sequence, _, _, vae_out = model.encode(
            input_ids=input_ids,
            attention_mask=attention_mask,
            sample_posterior=False,
        )
        emotion_init = vae_out.scalar_mu.detach()
        meaning_fixed = vae_out.vector_mu.detach()

    emotion_edit = nn.Parameter(emotion_init.clone())
    latent_optimizer = torch.optim.Adam([emotion_edit], lr=lr)
    losses: List[float] = []

    for _ in range(steps):
        latent_optimizer.zero_grad(set_to_none=True)

        if model.attention_source == "scalar_only":
            source_sequence = emotion_edit
        elif model.attention_source == "vector_only":
            source_sequence = meaning_fixed
        elif model.attention_source == "latent_full":
            source_sequence = torch.cat([emotion_edit, meaning_fixed], dim=-1)
        else:
            source_sequence = t5_encoder_sequence

        pooled, _ = model.scalar_attention_pool(
            source_sequence=source_sequence,
            scalar_sequence=emotion_edit,
            attention_mask=attention_mask,
        )
        logits = model.classification_logits(pooled)
        target_loss = masked_bce_with_logits(logits, desired_target, desired_mask)
        reg_loss = F.mse_loss(emotion_edit, emotion_init)
        total_loss = target_loss + (emotion_l2_weight * reg_loss)
        total_loss.backward()
        latent_optimizer.step()
        losses.append(float(total_loss.detach().cpu().item()))

    with torch.no_grad():
        edited_latent = torch.cat([emotion_edit, meaning_fixed], dim=-1)
        decoded_memory = model.memory_norm(model.vae_decoder(edited_latent) + model.residual_bottleneck(t5_encoder_sequence))
        if model.attention_source == "scalar_only":
            final_source_sequence = emotion_edit
        elif model.attention_source == "vector_only":
            final_source_sequence = meaning_fixed
        elif model.attention_source == "latent_full":
            final_source_sequence = torch.cat([emotion_edit, meaning_fixed], dim=-1)
        else:
            final_source_sequence = t5_encoder_sequence

        final_logits = model.classification_logits(
            model.scalar_attention_pool(
                source_sequence=final_source_sequence,
                scalar_sequence=emotion_edit,
                attention_mask=attention_mask,
            )[0]
        )
        final_probs = torch.sigmoid(final_logits)
        final_preds = (final_probs >= state.threshold).float()
        generated_text = generate_text_from_memory(
            model=model,
            tokenizer=ctx.tokenizer,
            decoder_memory=decoded_memory,
            encoder_attention_mask=attention_mask,
            prompt_config=ctx.prompt_config,
            max_new_tokens=ctx.data_config.max_length,
            num_beams=1,
            do_sample=False,
        )

    return {
        "edited_emotion_latents": emotion_edit.detach().cpu(),
        "fixed_meaning_latents": meaning_fixed.detach().cpu(),
        "loss_curve": losses,
        "final_logits": final_logits.detach().cpu(),
        "final_probs": final_probs.detach().cpu(),
        "final_preds": final_preds.detach().cpu(),
        "generated_text": generated_text,
    }


In [19]:
def format_label_names(names: Sequence[str]) -> str:
    if not names:
        return "(none)"
    return ", ".join(names)



def label_names_from_tensor(label_tensor: torch.Tensor, label_names: Sequence[str], threshold: float = 0.5) -> List[str]:
    active = torch.where(label_tensor.detach().cpu() >= threshold)[0].tolist()
    return [label_names[idx] for idx in active]



def predicted_names_from_logits(logits: torch.Tensor, label_names: Sequence[str], threshold: float) -> List[str]:
    probs = torch.sigmoid(logits.detach().cpu())
    active = torch.where(probs >= threshold)[0].tolist()
    return [label_names[idx] for idx in active]



def predict_emotions_for_text(state: TrainingState, ctx: ExperimentContext, text: str) -> List[str]:
    encoded = ctx.tokenizer(
        text,
        max_length=ctx.data_config.max_length,
        padding=True,
        truncation=True,
        return_tensors="pt",
    )
    input_ids = encoded["input_ids"].to(ctx.device)
    attention_mask = encoded["attention_mask"].to(ctx.device)
    with torch.no_grad():
        out = state.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=None,
            sample_posterior=ctx.experiment_config.sample_posterior_eval,
            classification_weight=ctx.loss_config.classification_weight,
            recon_weight=ctx.loss_config.recon_weight,
            kl_weight=ctx.loss_config.kl_weight,
        )
    return predicted_names_from_logits(out.classification_logits[0], ctx.emotion_names, state.threshold)



def pretty_print_text_block(title: str, text: str, width: int = 100) -> None:
    import textwrap

    print(title)
    wrapped = textwrap.fill(text, width=width, subsequent_indent="    ")
    print(f"    {wrapped}")



def print_generation_summary(tag: str, source_text: str, generated_text: str, predicted_emotions: Sequence[str]) -> None:
    copy_stats = summarize_copy_metrics([source_text], [generated_text])
    print(tag)
    pretty_print_text_block("  generated text:", generated_text)
    print(f"  predicted emotions on generated text: {format_label_names(predicted_emotions)}")
    print(
        "  copy metrics: "
        f"exact_match={copy_stats['exact_match']:.3f}, "
        f"token_f1={copy_stats['token_f1']:.3f}, "
        f"edit_similarity={copy_stats['edit_similarity']:.3f}"
    )



def run_epoch_monitor(
    state: TrainingState,
    ctx: ExperimentContext,
    dataset: Dataset,
    example_indices: Sequence[int],
    edit_targets: Sequence[Sequence[str]],
    epoch: int,
) -> None:
    state.model.eval()
    print("\n" + "=" * 120)
    print(f"Epoch {epoch} qualitative monitor on fixed validation examples")
    print("=" * 120)

    for slot, (example_idx, desired_labels) in enumerate(zip(example_indices, edit_targets), start=1):
        row = dataset[example_idx]
        text = row["text"]
        gold_names = label_names_from_tensor(row["labels"], ctx.emotion_names)

        encoded = ctx.tokenizer(
            text,
            max_length=ctx.data_config.max_length,
            padding=True,
            truncation=True,
            return_tensors="pt",
        )
        input_ids = encoded["input_ids"].to(ctx.device)
        attention_mask = encoded["attention_mask"].to(ctx.device)
        label_tensor = row["labels"].unsqueeze(0).to(ctx.device)

        with torch.no_grad():
            out = state.model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=label_tensor,
                sample_posterior=ctx.experiment_config.sample_posterior_eval,
                classification_weight=ctx.loss_config.classification_weight,
                recon_weight=ctx.loss_config.recon_weight,
                kl_weight=ctx.loss_config.kl_weight,
            )
            bypass_text = generate_text_from_memory(
                model=state.model,
                tokenizer=ctx.tokenizer,
                decoder_memory=out.t5_encoder_sequence,
                encoder_attention_mask=attention_mask,
                prompt_config=ctx.prompt_config,
                max_new_tokens=ctx.data_config.max_length,
                num_beams=1,
                do_sample=False,
            )[0]
            vae_text = generate_text_from_memory(
                model=state.model,
                tokenizer=ctx.tokenizer,
                decoder_memory=out.decoder_memory,
                encoder_attention_mask=attention_mask,
                prompt_config=ctx.prompt_config,
                max_new_tokens=ctx.data_config.max_length,
                num_beams=1,
                do_sample=False,
            )[0]

        input_pred_names = predicted_names_from_logits(out.classification_logits[0], ctx.emotion_names, state.threshold)
        bypass_pred_names = predict_emotions_for_text(state, ctx, bypass_text)
        vae_pred_names = predict_emotions_for_text(state, ctx, vae_text)

        desired_target, desired_mask = make_partial_desired_emotion_target(
            label_names=ctx.emotion_names,
            positive_labels=desired_labels,
        )
        edit_result = optimize_emotion_latents_for_target(
            state=state,
            ctx=ctx,
            input_ids=input_ids,
            attention_mask=attention_mask,
            desired_target=desired_target,
            desired_mask=desired_mask,
            steps=ctx.experiment_config.monitor_edit_steps,
            lr=3e-2,
            emotion_l2_weight=5e-3,
        )
        edited_text = edit_result["generated_text"][0]
        edited_head_pred_names = predicted_names_from_logits(
            edit_result["final_logits"][0],
            ctx.emotion_names,
            state.threshold,
        )
        edited_generated_pred_names = predict_emotions_for_text(state, ctx, edited_text)

        print("\n" + "-" * 120)
        print(f"Example {slot} | validation index={example_idx}")
        pretty_print_text_block("input text:", text)
        print(f"gold emotions: {format_label_names(gold_names)}")
        print(f"predicted emotions on input: {format_label_names(input_pred_names)}")
        print(
            "loss snapshot: "
            f"recon={float(out.loss_terms['reconstruction'].item()):.4f}, "
            f"kl={float(out.loss_terms['kl'].item()):.4f}, "
            f"cls={float(out.loss_terms['classification'].item()):.4f}"
        )

        print_generation_summary(
            tag="\nBypass path: encoder -> decoder",
            source_text=text,
            generated_text=bypass_text,
            predicted_emotions=bypass_pred_names,
        )
        print_generation_summary(
            tag="\nVAE path: encoder -> token VAE -> decoder",
            source_text=text,
            generated_text=vae_text,
            predicted_emotions=vae_pred_names,
        )

        print("\nEmotion editing")
        print(f"  requested target emotions: {format_label_names(desired_labels)}")
        pretty_print_text_block("  generated edited text:", edited_text)
        print(f"  head predictions after latent edit: {format_label_names(edited_head_pred_names)}")
        print(f"  predicted emotions on edited generated text: {format_label_names(edited_generated_pred_names)}")
        if edit_result['loss_curve']:
            print(f"  final edit objective: {edit_result['loss_curve'][-1]:.4f}")


## Training schedule

The training schedule is now intentionally simple.

- The VAE-side losses keep **constant weights** across epochs.
- LoRA stays frozen for the first five epochs.
- Starting at **epoch 6**, LoRA weights become trainable.
- Evaluation is run every epoch.
- The qualitative monitor is also run every epoch so you can inspect what the numbers mean.

This makes the logs easier to interpret because any major behavioral change around epoch 6 comes from enabling LoRA rather than from several loss ramps changing at the same time.


In [20]:
def current_loss_weights(epoch: int, ctx: ExperimentContext) -> Dict[str, float]:
    return {
        "classification_weight": ctx.loss_config.classification_weight,
        "recon_weight": ctx.loss_config.recon_weight,
        "kl_weight": ctx.loss_config.kl_weight,
        "tc_weight": ctx.loss_config.tc_weight,
        "copy_weight": ctx.loss_config.copy_weight if epoch >= ctx.schedule_config.copy_loss_start_epoch else 0.0,
        "vector_adv_weight": ctx.loss_config.vector_adv_weight,
        "vector_sep_weight": ctx.loss_config.vector_sep_weight,
    }



def ensure_lora_schedule(epoch: int, state: TrainingState, ctx: ExperimentContext) -> None:
    enable_lora = epoch >= ctx.schedule_config.lora_start_epoch
    set_lora_trainable(state.model.shared_t5, enabled=enable_lora)



def slim_history_row(epoch: int, split_name: str, metrics: Dict[str, Any]) -> Dict[str, Any]:
    keep_keys = [
        "threshold",
        "avg_total_loss",
        "avg_base_loss",
        "avg_recon",
        "avg_kl",
        "avg_cls",
        "avg_tc",
        "avg_copy",
        "avg_vec_adv",
        "avg_sep",
        "weighted_recon",
        "weighted_kl",
        "weighted_cls",
        "weighted_tc",
        "weighted_copy",
        "weighted_vec_adv",
        "weighted_sep",
        "micro_f1",
        "macro_f1",
        "weighted_f1",
        "subset_accuracy",
        "hamming_acc",
        "jaccard_micro",
        "average_precision_micro",
        "label_ranking_average_precision",
    ]
    row = {"epoch": epoch, "split": split_name}
    for key in keep_keys:
        if key in metrics:
            value = metrics[key]
            row[key] = float(value) if isinstance(value, (int, float, np.floating, np.integer)) else value
    return row



def save_checkpoint(
    path: Path,
    state: TrainingState,
    ctx: ExperimentContext,
    epoch: int,
    val_metrics: Optional[Dict[str, Any]] = None,
    test_metrics: Optional[Dict[str, Any]] = None,
) -> None:
    payload = {
        "epoch": epoch,
        "step": state.step,
        "threshold": state.threshold,
        "best_val_micro_f1": state.best_val_micro_f1,
        "model_state_dict": state.model.state_dict(),
        "discriminator_state_dict": None if state.discriminator is None else state.discriminator.state_dict(),
        "vector_adversary_state_dict": None if state.vector_adversary is None else state.vector_adversary.state_dict(),
        "optimizer_state_dict": state.optimizer.state_dict(),
        "disc_optimizer_state_dict": None if state.disc_optimizer is None else state.disc_optimizer.state_dict(),
        "vector_adv_optimizer_state_dict": None if state.vector_adv_optimizer is None else state.vector_adv_optimizer.state_dict(),
        "lr_scheduler_state_dict": state.lr_scheduler.state_dict(),
        "history": state.history,
        "configs": {
            "data_config": asdict(ctx.data_config),
            "prompt_config": asdict(ctx.prompt_config),
            "model_config": asdict(ctx.model_config),
            "loss_config": asdict(ctx.loss_config),
            "schedule_config": asdict(ctx.schedule_config),
            "experiment_config": asdict(ctx.experiment_config),
        },
        "emotion_names": list(ctx.emotion_names),
        "val_metrics": None if val_metrics is None else {k: v for k, v in to_serializable(val_metrics).items() if k not in {"logits", "labels", "probs", "preds", "classification_report_text"}},
        "test_metrics": None if test_metrics is None else {k: v for k, v in to_serializable(test_metrics).items() if k not in {"logits", "labels", "probs", "preds", "classification_report_text"}},
    }
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(payload, path)


In [21]:
def train_one_epoch(epoch: int, state: TrainingState, ctx: ExperimentContext, weights: Dict[str, float]) -> Dict[str, Any]:
    state.model.train()
    if state.vector_adversary is not None:
        state.vector_adversary.train()
    epoch_total = 0.0
    epoch_base = 0.0
    epoch_recon = 0.0
    epoch_kl = 0.0
    epoch_cls = 0.0
    epoch_tc = 0.0
    epoch_copy = 0.0
    epoch_vec_adv = 0.0
    epoch_sep = 0.0
    epoch_disc = 0.0
    epoch_logits = []
    epoch_labels = []
    batches = 0

    train_bar = tqdm(train_loader, desc=f"epoch {epoch}/{SCHEDULE_CONFIG.num_epochs} [train]", leave=False)
    for batch in train_bar:
        state.optimizer.zero_grad(set_to_none=True)
        if state.vector_adv_optimizer is not None:
            state.vector_adv_optimizer.zero_grad(set_to_none=True)

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        texts = batch["texts"]

        out = state.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
            sample_posterior=EXPERIMENT_CONFIG.sample_posterior_train,
            classification_weight=weights["classification_weight"],
            recon_weight=weights["recon_weight"],
            kl_weight=weights["kl_weight"],
        )

        tc_loss = out.base_loss.new_zeros(())
        tc_for_disc = None
        if state.discriminator is not None:
            set_requires_grad(state.discriminator, False)
            tc_inputs = pooled_latents_for_tc(out.vae, attention_mask, ctx.loss_config.tc_subspace)
            tc_loss = factorvae_tc_loss(state.discriminator, tc_inputs)

        copy_loss = out.base_loss.new_zeros(())
        if weights["copy_weight"] > 0.0:
            copy_loss = compute_copy_loss_from_memory(
                model=state.model,
                tokenizer=ctx.tokenizer,
                decoder_memory=out.decoder_memory,
                encoder_attention_mask=attention_mask,
                texts=texts,
                prompt_config=ctx.prompt_config,
                device=ctx.device,
            )

        vector_summary = masked_mean(out.vae.vector_mu, attention_mask)
        scalar_summary = masked_mean(out.vae.scalar_mu, attention_mask)

        vec_adv_loss = out.base_loss.new_zeros(())
        if state.vector_adversary is not None and weights["vector_adv_weight"] > 0.0:
            _, vec_adv_loss = vector_adversarial_loss(
                vector_adversary=state.vector_adversary,
                vector_summary=vector_summary,
                labels=labels,
                pos_weight=ctx.pos_weight,
            )

        sep_loss = out.base_loss.new_zeros(())
        if weights["vector_sep_weight"] > 0.0:
            sep_loss = cross_covariance_penalty(scalar_summary, vector_summary)

        total_loss = (
            out.base_loss
            + (weights["tc_weight"] * tc_loss)
            + (weights["copy_weight"] * copy_loss)
            + (weights["vector_adv_weight"] * vec_adv_loss)
            + (weights["vector_sep_weight"] * sep_loss)
        )
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(state.model.parameters(), max_norm=SCHEDULE_CONFIG.max_grad_norm)
        if state.vector_adversary is not None:
            torch.nn.utils.clip_grad_norm_(state.vector_adversary.parameters(), max_norm=SCHEDULE_CONFIG.max_grad_norm)
        state.optimizer.step()
        if state.vector_adv_optimizer is not None:
            state.vector_adv_optimizer.step()
        state.lr_scheduler.step()

        disc_loss = out.base_loss.new_zeros(())
        if state.discriminator is not None and state.disc_optimizer is not None:
            set_requires_grad(state.discriminator, True)
            with torch.no_grad():
                disc_forward = state.model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    sample_posterior=True,
                    classification_weight=weights["classification_weight"],
                    recon_weight=weights["recon_weight"],
                    kl_weight=weights["kl_weight"],
                    labels=labels,
                )
                tc_for_disc = pooled_latents_for_tc(disc_forward.vae, attention_mask, ctx.loss_config.tc_subspace)

            if tc_for_disc is not None and tc_for_disc.size(0) >= 2:
                state.disc_optimizer.zero_grad(set_to_none=True)
                disc_loss = discriminator_loss(state.discriminator, tc_for_disc)
                disc_loss.backward()
                state.disc_optimizer.step()

        epoch_total += float(total_loss.detach().cpu().item())
        epoch_base += float(out.base_loss.detach().cpu().item())
        epoch_recon += float(out.loss_terms["reconstruction"].item())
        epoch_kl += float(out.loss_terms["kl"].item())
        epoch_cls += float(out.loss_terms["classification"].item())
        epoch_tc += float(tc_loss.detach().cpu().item())
        epoch_copy += float(copy_loss.detach().cpu().item())
        epoch_vec_adv += float(vec_adv_loss.detach().cpu().item())
        epoch_sep += float(sep_loss.detach().cpu().item())
        epoch_disc += float(disc_loss.detach().cpu().item())
        epoch_logits.append(out.classification_logits.detach().cpu())
        epoch_labels.append(labels.detach().cpu())
        batches += 1
        state.step += 1

        train_bar.set_postfix(
            loss=f"{epoch_total / max(batches, 1):.4f}",
            cls=f"{epoch_cls / max(batches, 1):.4f}",
            recon=f"{epoch_recon / max(batches, 1):.4f}",
            kl=f"{epoch_kl / max(batches, 1):.4f}",
            tc=f"{epoch_tc / max(batches, 1):.4f}",
            copy=f"{epoch_copy / max(batches, 1):.4f}",
            adv=f"{epoch_vec_adv / max(batches, 1):.4f}",
            sep=f"{epoch_sep / max(batches, 1):.4f}",
            lr=f"{state.optimizer.param_groups[0]['lr']:.2e}",
        )

        del out, total_loss, copy_loss, tc_loss, vec_adv_loss, sep_loss, disc_loss, input_ids, attention_mask, labels
        if tc_for_disc is not None:
            del tc_for_disc

    train_logits = torch.cat(epoch_logits, dim=0)
    train_labels = torch.cat(epoch_labels, dim=0)
    train_metrics = multilabel_metrics(
        logits=train_logits,
        labels=train_labels,
        threshold=state.threshold,
        label_names=ctx.emotion_names,
    )
    train_metrics.update(
        {
            "avg_total_loss": epoch_total / max(batches, 1),
            "avg_base_loss": epoch_base / max(batches, 1),
            "avg_recon": epoch_recon / max(batches, 1),
            "avg_kl": epoch_kl / max(batches, 1),
            "avg_cls": epoch_cls / max(batches, 1),
            "avg_tc": epoch_tc / max(batches, 1),
            "avg_copy": epoch_copy / max(batches, 1),
            "avg_vec_adv": epoch_vec_adv / max(batches, 1),
            "avg_sep": epoch_sep / max(batches, 1),
            "avg_disc": epoch_disc / max(batches, 1),
            "weighted_recon": weights["recon_weight"] * (epoch_recon / max(batches, 1)),
            "weighted_kl": weights["kl_weight"] * (epoch_kl / max(batches, 1)),
            "weighted_cls": weights["classification_weight"] * (epoch_cls / max(batches, 1)),
            "weighted_tc": weights["tc_weight"] * (epoch_tc / max(batches, 1)),
            "weighted_copy": weights["copy_weight"] * (epoch_copy / max(batches, 1)),
            "weighted_vec_adv": weights["vector_adv_weight"] * (epoch_vec_adv / max(batches, 1)),
            "weighted_sep": weights["vector_sep_weight"] * (epoch_sep / max(batches, 1)),
        }
    )
    release_memory()
    return train_metrics


### Run training and evaluation

This loop prints four things every epoch:

- the active loss weights,
- train metrics,
- calibration and validation metrics,
- the qualitative monitor on the fixed validation examples.


## Preflight check before training

This cell confirms that the notebook has every required helper loaded **before** the training loop starts. It also prints the current batch sizes so VRAM usage is explicit.


In [22]:
required_symbols = [
    "train_one_epoch",
    "evaluate_loader",
    "multilabel_metrics",
    "tune_global_threshold_for_micro_f1",
    "summarize_copy_metrics",
    "run_epoch_monitor",
    "optimize_emotion_latents_for_target",
    "generate_text_from_memory",
    "VectorEmotionAdversary",
    "vector_adversarial_loss",
    "cross_covariance_penalty",
    "pooled_latents_for_tc",
]
missing_symbols = [name for name in required_symbols if name not in globals()]
assert not missing_symbols, f"Missing definitions before training: {missing_symbols}"
print("preflight ok")
print(f"train_batch_size={DATA_CONFIG.train_batch_size}, eval_batch_size={DATA_CONFIG.eval_batch_size}")


preflight ok
train_batch_size=8, eval_batch_size=8


In [ ]:
for epoch in range(1, SCHEDULE_CONFIG.num_epochs + 1):
    ensure_lora_schedule(epoch=epoch, state=state, ctx=ctx)
    weights = current_loss_weights(epoch=epoch, ctx=ctx)

    print(
        f"\n[epoch {epoch}] weights="
        f"{{'cls': {weights['classification_weight']:.2f}, 'recon': {weights['recon_weight']:.2f}, "
        f"'kl': {weights['kl_weight']:.4f}, 'tc': {weights['tc_weight']:.4f}, 'copy': {weights['copy_weight']:.4f}, "
        f"'vec_adv': {weights['vector_adv_weight']:.4f}, 'sep': {weights['vector_sep_weight']:.4f}}} "
        f"lora_enabled={epoch >= SCHEDULE_CONFIG.lora_start_epoch}"
    )

    train_metrics = train_one_epoch(epoch=epoch, state=state, ctx=ctx, weights=weights)
    state.history.append(slim_history_row(epoch=epoch, split_name="train", metrics=train_metrics))
    summarize_eval_metrics(prefix=f"[epoch {epoch}] train ", metrics=train_metrics)

    threshold_metrics = evaluate_loader(
        state=state,
        ctx=ctx,
        data_loader=threshold_loader,
        desc=f"epoch {epoch}/{SCHEDULE_CONFIG.num_epochs} [threshold]",
        tune_threshold=True,
        weights=weights,
    )
    state.threshold = float(threshold_metrics["threshold"])
    summarize_eval_metrics(prefix=f"[epoch {epoch}] calib ", metrics=threshold_metrics)
    state.history.append(slim_history_row(epoch=epoch, split_name="threshold_tune", metrics=threshold_metrics))

    val_metrics = evaluate_loader(
        state=state,
        ctx=ctx,
        data_loader=val_loader,
        desc=f"epoch {epoch}/{SCHEDULE_CONFIG.num_epochs} [val]",
        tune_threshold=False,
        weights=weights,
    )
    summarize_eval_metrics(prefix=f"[epoch {epoch}] val   ", metrics=val_metrics)
    state.history.append(slim_history_row(epoch=epoch, split_name="validation", metrics=val_metrics))
    save_json(output_dir / "history.json", state.history)

    release_memory()
    run_epoch_monitor(
        state=state,
        ctx=ctx,
        dataset=val_dataset,
        example_indices=monitor_example_indices,
        edit_targets=monitor_target_labels,
        epoch=epoch,
    )

    if float(val_metrics["micro_f1"]) > state.best_val_micro_f1:
        state.best_val_micro_f1 = float(val_metrics["micro_f1"])
        save_checkpoint(
            path=output_dir / "best_checkpoint.pt",
            state=state,
            ctx=ctx,
            epoch=epoch,
            val_metrics=val_metrics,
        )
        print(f"saved new best checkpoint at epoch {epoch} with val micro-F1={state.best_val_micro_f1:.4f}")

    save_checkpoint(
        path=output_dir / f"checkpoint_epoch_{epoch:03d}.pt",
        state=state,
        ctx=ctx,
        epoch=epoch,
        val_metrics=val_metrics,
    )


release_memory()



[epoch 1] weights={'cls': 1.00, 'recon': 1.00, 'kl': 1.0000, 'tc': 6.8000, 'copy': 0.0000} lora_enabled=False


epoch 1/30 [train]:   0%|          | 0/5155 [00:00<?, ?it/s]

[epoch 1] train loss=0.9451 raw(recon=0.0111, kl=0.0191, cls=0.9148, tc=0.0000, copy=0.0000) weighted(recon=0.0111, kl=0.0191, cls=0.9148, tc=0.0001, copy=0.0000) threshold=0.500 micro_f1=0.1493 macro_f1=0.0835 weighted_f1=0.1857 subset_acc=0.0048 hamming_acc=0.8214 jaccard_micro=0.0807 ap_micro=0.0969 lrap=0.2632


epoch 1/30 [threshold]:   0%|          | 0/272 [00:00<?, ?it/s]

[epoch 1] calib loss=0.8398 raw(recon=0.0112, kl=0.0144, cls=0.8215, tc=-0.0011, copy=0.0000) weighted(recon=0.0112, kl=0.0144, cls=0.8215, tc=-0.0074, copy=0.0000) threshold=0.529 micro_f1=0.2501 macro_f1=0.1243 weighted_f1=0.2636 subset_acc=0.1594 hamming_acc=0.8789 jaccard_micro=0.1429 ap_micro=0.1702 lrap=0.4595


...

epoch 6/30 [threshold]:   0%|          | 0/272 [00:00<?, ?it/s]

[epoch 6] calib loss=0.5621 raw(recon=0.0108, kl=0.0095, cls=0.5793, tc=-0.0055, copy=0.0000) weighted(recon=0.0108, kl=0.0095, cls=0.5793, tc=-0.0376, copy=0.0000) threshold=0.588 micro_f1=0.3626 macro_f1=0.3029 weighted_f1=0.3999 subset_acc=0.0908 hamming_acc=0.9072 jaccard_micro=0.2214 ap_micro=0.3604 lrap=0.6109


epoch 6/30 [val]:   0%|          | 0/679 [00:00<?, ?it/s]

[epoch 6] val   loss=0.5454 raw(recon=0.0108, kl=0.0094, cls=0.5627, tc=-0.0055, copy=0.0000) weighted(recon=0.0108, kl=0.0094, cls=0.5627, tc=-0.0376, copy=0.0000) threshold=0.588 micro_f1=0.3698 macro_f1=0.3038 weighted_f1=0.4125 subset_acc=0.0929 hamming_acc=0.9088 jaccard_micro=0.2268 ap_micro=0.3874 lrap=0.6284

Epoch 6 qualitative monitor on fixed validation examples

------------------------------------------------------------------------------------------------------------------------
Example 1 | validation index=547
input text:
    Meh good introduction. Sadly I am a pro philosopher so know all of this
gold emotions: admiration
predicted emotions on input: (none)
loss snapshot: recon=0.0104, kl=0.0099, cls=0.8532

Bypass path: encoder -> decoder
  generated text:
    i am a pro philosopher
  predicted emotions on generated text: approval, neutral
  copy metrics: exact_match=0.000, token_f1=0.526, edit_similarity=0.473

VAE path: encoder -> token VAE -> decoder
  generated text

epoch 7/30 [train]:   0%|          | 0/5155 [00:00<?, ?it/s]

## Compact training history view

This small table makes it easier to scan epoch-by-epoch progress after the full run finishes.


In [ ]:
try:
    import pandas as pd

    history_df = pd.DataFrame(state.history)
    epoch_summary = history_df[history_df["split"].isin(["train", "validation"])].copy()
    epoch_summary = epoch_summary[[
        "epoch",
        "split",
        "avg_total_loss",
        "avg_kl",
        "weighted_kl",
        "avg_copy",
        "weighted_copy",
        "micro_f1",
        "macro_f1",
        "threshold",
    ]]
    epoch_summary
except Exception as exc:
    print("Could not build pandas summary:", exc)


## Final validation and test evaluation

After training finishes, the notebook reloads the best validation checkpoint and evaluates it once more.

The test split is still kept for the end.


In [ ]:
best_checkpoint = torch.load(output_dir / "best_checkpoint.pt", map_location=device)
state.model.load_state_dict(best_checkpoint["model_state_dict"])
if state.discriminator is not None and best_checkpoint.get("discriminator_state_dict") is not None:
    state.discriminator.load_state_dict(best_checkpoint["discriminator_state_dict"])
if state.vector_adversary is not None and best_checkpoint.get("vector_adversary_state_dict") is not None:
    state.vector_adversary.load_state_dict(best_checkpoint["vector_adversary_state_dict"])
state.threshold = float(best_checkpoint["threshold"])

final_weights = current_loss_weights(epoch=SCHEDULE_CONFIG.num_epochs, ctx=ctx)

final_threshold_metrics = evaluate_loader(
    state=state,
    ctx=ctx,
    data_loader=threshold_loader,
    desc="final threshold set",
    tune_threshold=False,
    weights=final_weights,
)
final_val_metrics = evaluate_loader(
    state=state,
    ctx=ctx,
    data_loader=val_loader,
    desc="final validation",
    tune_threshold=False,
    weights=final_weights,
)
final_test_metrics = evaluate_loader(
    state=state,
    ctx=ctx,
    data_loader=test_loader,
    desc="final test",
    tune_threshold=False,
    weights=final_weights,
)

summarize_eval_metrics(prefix="[final] calib ", metrics=final_threshold_metrics)
summarize_eval_metrics(prefix="[final] val   ", metrics=final_val_metrics)
summarize_eval_metrics(prefix="[final] test  ", metrics=final_test_metrics)
print("\n[final] test classification report\n")
print(final_test_metrics["classification_report_text"])

state.history.append(slim_history_row(epoch=SCHEDULE_CONFIG.num_epochs, split_name="test", metrics=final_test_metrics))
save_json(output_dir / "history.json", state.history)
save_json(
    output_dir / "final_metrics.json",
    {
        "threshold_tune": {k: v for k, v in to_serializable(final_threshold_metrics).items() if k not in {"logits", "labels", "probs", "preds", "classification_report_text"}},
        "validation": {k: v for k, v in to_serializable(final_val_metrics).items() if k not in {"logits", "labels", "probs", "preds", "classification_report_text"}},
        "test": {k: v for k, v in to_serializable(final_test_metrics).items() if k not in {"logits", "labels", "probs", "preds", "classification_report_text"}},
    },
)
save_checkpoint(
    path=output_dir / "final_checkpoint.pt",
    state=state,
    ctx=ctx,
    epoch=SCHEDULE_CONFIG.num_epochs,
    val_metrics=final_val_metrics,
    test_metrics=final_test_metrics,
)

release_memory()


## Optional prompt experiment after training

This experiment no longer prepends anything to the source text. It compares prompt candidates directly against the true source text.


In [ ]:

prompt_eval_texts = [test_dataset[i]["text"] for i in range(min(EXPERIMENT_CONFIG.prompt_eval_num_examples, len(test_dataset)))]
prompt_results = evaluate_prompt_candidates(
    state=state,
    ctx=ctx,
    texts=prompt_eval_texts,
    prompt_candidates=EXPERIMENT_CONFIG.prompt_candidates,
    use_vae_memory=True,
    max_new_tokens=DATA_CONFIG.max_length,
)
prompt_results

release_memory()


## Final qualitative walkthrough on the best checkpoint

This reruns the same fixed qualitative monitor after the best checkpoint has been reloaded.

That gives one clean, final readout without scrolling back through every epoch.


In [ ]:
run_epoch_monitor(
    state=state,
    ctx=ctx,
    dataset=val_dataset,
    example_indices=monitor_example_indices,
    edit_targets=monitor_target_labels,
    epoch=best_checkpoint["epoch"],
)

release_memory()


## Optional single-example latent editing demo

The monitor above already runs latent editing every epoch.

This extra cell is only for ad hoc inspection of one chosen example and one chosen target emotion set.


In [ ]:
example_idx = monitor_example_indices[0]
example_text = val_dataset[example_idx]["text"]
desired_labels = monitor_target_labels[0]
desired_target, desired_mask = make_partial_desired_emotion_target(
    label_names=emotion_names,
    positive_labels=desired_labels,
)

encoded = tokenizer(
    example_text,
    max_length=DATA_CONFIG.max_length,
    padding=True,
    truncation=True,
    return_tensors="pt",
)
edit_result = optimize_emotion_latents_for_target(
    state=state,
    ctx=ctx,
    input_ids=encoded["input_ids"],
    attention_mask=encoded["attention_mask"],
    desired_target=desired_target,
    desired_mask=desired_mask,
    steps=80,
    lr=3e-2,
    emotion_l2_weight=5e-3,
)

pred_ids = torch.where(edit_result["final_preds"][0] > 0.5)[0].tolist()
pred_names = [emotion_names[idx] for idx in pred_ids]

print("source text:", example_text)
print("desired labels:", desired_labels)
print("predicted labels after edit:", pred_names)
print("generated text after edit:", edit_result["generated_text"][0])
print("final optimization loss:", edit_result["loss_curve"][-1])


## Ablation knobs retained on purpose

The following settings are kept because they are useful ablations, even though they are not the recommended default:

- `classifier_mode="joint_mlp"`
- `pooling_mode="joint_scalar_vector"`
- `attention_source in {"vector_only", "latent_full", "encoder_sequence"}`

Those settings should be treated as experiments that deliberately weaken or change the intended scalar-emotion mapping, not as equivalent defaults.
